Alternative Preprocessing using RegBN

Individual assignment by Lyder Samnøy

Requires pytorch CUDA-support https://pytorch.org/get-started/locally/

Imports + paths

In [1]:
import sys
from pathlib import Path
import torch
import pandas as pd

# Import RegBN
sys.path.append('.')
from RegBN import RegBN

# Paths (same as Preprocessing.ipynb)
TrainingPath = Path("..") / "Training data"
ValidationPath = Path("..") / "Evaluation data"
SavePath = Path("..") / "Processed data"

Load processed outputs from Preprocessing

In [2]:
# Load already processed tabular data
train_tabular = pd.read_csv(SavePath / "processed_data.csv")
eval_tabular  = pd.read_csv(SavePath / "processed_eval.csv")

print("Train shape:", train_tabular.shape)
print("Eval shape:", eval_tabular.shape)

# Load processed image tensors
img_dict = {}
for img_file in (SavePath / "processed_composite").glob("*.pt"):
    img_dict[img_file.name] = torch.load(img_file)

print(f"Loaded {len(img_dict)} processed images")

Train shape: (1024, 20)
Eval shape: (1024, 19)
Loaded 2048 processed images


Define numeric columns

In [3]:
numeric_cols = [
    'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km',
    'access_to_airport', 'access_to_highway', 'access_to_port',
    'access_to_railway', 'country', 'landlocked', 'flood_risk_class',
    'quarter_label', 'region_economic_classification',
    'seismic_hazard_zone', 'tornadoes_wind_risk',
    'tropical_cyclone_wind_risk'
]

Infer dimensions

In [4]:
sample_img = list(img_dict.values())[0]

c_s, h_s, w_s = sample_img['sentinel'].shape
c_v, h_v, w_v = sample_img['viirs'].shape

f_num_channels = c_s * h_s * w_s + c_v * h_v * w_v
g_num_channels = len(numeric_cols)

print(f"f_num_channels: {f_num_channels}, g_num_channels: {g_num_channels}")

f_num_channels: 652288, g_num_channels: 15


Instantiate RegBN (CUDA REQUIRED)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

regbn = RegBN(
    f_num_channels=f_num_channels,
    g_num_channels=g_num_channels,
    f_layer_dim=[],
    g_layer_dim=[],
    normalize_input=True,
    normalize_output=True,
    affine=False,
    verbose=True
)

regbn = regbn.to(device)

Build training tensors

In [6]:
train_f_list = []
train_g_list = []

for _, row in train_tabular.iterrows():
    img = img_dict[row['processed_imgs']]

    f = torch.cat([
        img['sentinel'].flatten(),
        img['viirs'].flatten()
    ])

    g = torch.tensor(row[numeric_cols].values.astype("float32"))

    train_f_list.append(f)
    train_g_list.append(g)

train_f = torch.stack(train_f_list).to(device)
train_g = torch.stack(train_g_list).to(device)

print(f"Train f shape: {train_f.shape}, train g shape: {train_g.shape}")

Train f shape: torch.Size([1024, 652288]), train g shape: torch.Size([1024, 15])


Train RegBN

In [7]:
num_epochs = 5
batch_size = 64
num_batches = (len(train_f) + batch_size - 1) // batch_size

regbn.train()

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    for i in range(0, len(train_f), batch_size):
        f_batch = train_f[i:i+batch_size]
        g_batch = train_g[i:i+batch_size]

        regbn(
            f_batch,
            g_batch,
            is_training=True,
            n_epoch=epoch,
            steps_per_epoch=num_batches
        )

# Move back to CPU for inference
regbn = regbn.cpu()
train_f = train_f.cpu()
train_g = train_g.cpu()

torch.save(regbn.state_dict(), SavePath / "regbn_model.pth")
print("RegBN model saved.")

Epoch 1/5
Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5
RegBN model saved.


Normalize function

In [8]:
def normalize_data(df, img_dict, regbn, numeric_cols,
                   c_s, h_s, w_s, c_v, h_v, w_v):

    normalized_rows = []
    regbn.eval()

    with torch.no_grad():
        for _, row in df.iterrows():
            img = img_dict[row['processed_imgs']]

            f = torch.cat([
                img['sentinel'].flatten(),
                img['viirs'].flatten()
            ]).unsqueeze(0).float()

            g = torch.tensor(
                row[numeric_cols].values.astype("float32")
            ).unsqueeze(0)

            f_n, g_n = regbn(f, g, is_training=False)

            f_n = f_n.squeeze(0)
            sentinel_size = c_s * h_s * w_s

            sentinel_n = f_n[:sentinel_size].reshape(c_s, h_s, w_s)
            viirs_n = f_n[sentinel_size:].reshape(c_v, h_v, w_v)

            img_dict[row['processed_imgs']] = {
                'sentinel': sentinel_n,
                'viirs': viirs_n
            }

            new_row = row.copy()
            new_row[numeric_cols] = g_n.squeeze(0).numpy()

            normalized_rows.append(new_row)

    return pd.DataFrame(normalized_rows)

Apply normalization

In [10]:
print("Normalizing training data...")

train_tabular_normalized = normalize_data(
    train_tabular,
    img_dict,
    regbn,
    numeric_cols,
    c_s, h_s, w_s,
    c_v, h_v, w_v
)

train_tabular_normalized.to_csv(
    SavePath / "processed_data_regbn.csv",
    index=False
)

for name, img in img_dict.items():
    torch.save(img, SavePath / "processed_composite_regbn" / name)

print("Normalized data saved.")

Normalizing training data...
Normalized data saved.


Validation

In [ ]:
import torch
import numpy as np

def validate_regbn(train_f, train_g, regbn, batch_size=256):
    regbn.eval()

    total_energy_before = 0.0
    total_energy_after = 0.0

    f_before_all = []
    f_after_all = []
    g_all = []

    with torch.no_grad():
        for i in range(0, len(train_f), batch_size):
            f_batch = train_f[i:i+batch_size]
            g_batch = train_g[i:i+batch_size]

            f_n, g_n = regbn(f_batch, g_batch, is_training=False)

            total_energy_before += (f_batch**2).sum().item()
            total_energy_after  += (f_n**2).sum().item()

            f_before_all.append(f_batch)
            f_after_all.append(f_n)
            g_all.append(g_batch)

    f_before = torch.cat(f_before_all, dim=0)
    f_after  = torch.cat(f_after_all, dim=0)
    g_all    = torch.cat(g_all, dim=0)

    # --- BASIC METRICS ---
    energy_ratio = total_energy_after / total_energy_before
    energy_removed = 1 - energy_ratio

    mean_before = f_before.abs().mean().item()
    mean_after  = f_after.abs().mean().item()

    # --- RELATIVE CHANGE (IMPORTANT) ---
    relative_change = (
        (f_before - f_after).norm(dim=1) /
        (f_before.norm(dim=1) + 1e-6)
    ).mean().item()

    # --- PROJECTION MAGNITUDE ---
    projection_magnitude = (f_before - f_after).abs().mean().item()

    # --- CORRELATION ---
    def compute_corr(f, g):
        f_centered = f - f.mean(0)
        g_centered = g - g.mean(0)

        cov = torch.mm(f_centered.T, g_centered) / (f.shape[0] - 1)

        f_std = f_centered.std(0) + 1e-6
        g_std = g_centered.std(0) + 1e-6

        corr = cov / (f_std.unsqueeze(1) * g_std.unsqueeze(0))
        return corr.abs().mean().item()

    f_before_flat = f_before.view(f_before.shape[0], -1)
    f_after_flat  = f_after.view(f_after.shape[0], -1)

    corr_before = compute_corr(f_before_flat, g_all)
    corr_after  = compute_corr(f_after_flat, g_all)

    # --- W DIAGNOSTICS ---
    W_norm = regbn.W.norm().item()
    W_mean = regbn.W.abs().mean().item()

    return {
        "energy_removed_%": energy_removed * 100,
        "energy_ratio": energy_ratio,

        "mean_abs_before": mean_before,
        "mean_abs_after": mean_after,

        "relative_change_%": relative_change * 100,
        "projection_magnitude": projection_magnitude,

        "corr_before": corr_before,
        "corr_after": corr_after,
        "corr_reduction_%": (1 - corr_after / corr_before) * 100,

        "W_norm": W_norm,
        "W_mean_abs": W_mean
    }


# Run validation
results = validate_regbn(train_f.cpu(), train_g.cpu(), regbn)

print("\n=== RegBN Validation (Improved) ===")
for k, v in results.items():
    print(f"{k}: {v:.6f}")

print("Mean std of g:", train_g.std(0).mean().item())

f_before std: 0.9999900460243225
f_after std: 0.9999940991401672

=== RegBN Validation (Improved) ===
energy_removed_%: -0.000817
energy_ratio: 1.000008
mean_abs_before: 0.793783
mean_abs_after: 0.757200
relative_change_%: 22.298329
projection_magnitude: 0.189090
corr_before: 0.066880
corr_after: 0.058557
corr_reduction_%: 12.445449
W_norm: 75.764450
W_mean_abs: 0.017289
Mean std of g: 0.44890275597572327
